# Wind Speed & Direction — Reproducible Metocean Analysis

**تحلیل آماری حداکثر سرعت روزانه باد و جهت متناظر با آن**

این نوت‌بوک، داده اکسل را می‌خواند، در صورت لزوم از حالت افقی به روزانه تبدیل می‌کند، آمار سرعت و جهت را محاسبه می‌کند و نمودارهای گزارش را با خروجی **PNG (300 dpi)** و **PDF برداری** ذخیره می‌کند.

**شروع:** ابتدا `requirements.txt` را نصب کن، بعد فقط مسیر `INPUT_FILE` را در سلول تنظیمات تغییر بده. فایل داده خام را بدون مجوز در GitHub منتشر نکن. تابع‌های محاسباتی در فایل همراه `wind_analysis.py` قرار دارند.

## 1. تنظیمات کاربر — این سلول را ویرایش کن

In [ ]:
from pathlib import Path
import wind_analysis as wa

# !!! EDIT THE PATH TO YOUR INPUT DATA FILE HERE !!!
INPUT_FILE = Path("data/wind_data.xlsx")
# Or use your own full Windows path (uncomment the next line):
# INPUT_FILE = Path(r"C:\Users\YOUR_NAME\Desktop\wind\wind speed and direction.xlsx")

# 'wide': Persian month/year headers + paired Dir/Wind columns (default)
# 'long': already normalized Year/Month/Day/Direction/WindSpeed columns
INPUT_LAYOUT = "wide"
INPUT_SHEET = "Sheet1"
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Analysis settings - feel free to edit
wa.STRONG_WIND_THRESHOLD = 12.0  # strictly > 12 m/s
wa.DIRECTION_BIN_DEG = 10
wa.SPEED_BIN_EDGES = [0, 3, 6, 9, 12, 15, 20]
# Update this set when analyzing Persian calendar years outside 1389–1404:
wa.PERSIAN_LEAP_YEARS = {1391, 1395, 1399, 1403}

print('Input:', INPUT_FILE.resolve())
print('Output folder:', OUTPUT_DIR.resolve())

## 2. خواندن و مرتب‌سازی داده‌ها

In [ ]:
df_long, df_speed, df_dir, quality = wa.prepare_data(
    input_file=INPUT_FILE,
    input_layout=INPUT_LAYOUT,
    sheet_name=INPUT_SHEET,
)
print('First 5 daily records:')
display(df_long.head())
print('Data quality:')
display(quality.to_frame('Value'))

**توضیح:** `df_long` تمام روزهای تقویمی (حتی روزهای فاقد داده) را نگه می‌دارد. `df_speed` شامل روزهای دارای سرعت معتبر است و `df_dir` فقط روزهای دارای سرعت مثبت و جهت معتبر را برای گلباد و تحلیل جهت استفاده می‌کند. جهت‌های صفر و ۳۶۰ به شمال یکسان‌سازی می‌شوند.

## 3. محاسبه آمارهای اصلی

In [ ]:
results = wa.analyze_wind(df_long, df_speed, df_dir)
print('Wind speed statistics (daily maxima):')
display(results['stats'].round(2).to_frame('Value'))
print('Dominant directions:')
display(results['direction'].sort_values('All count', ascending=False).head(10).round(2))
print('Top 10 daily maxima:')
display(results['top10'])

## 4. هیستوگرام سرعت و منحنی فراگذشت

In [ ]:
wa.plot_speed_distribution(df_speed, OUTPUT_DIR)
wa.plot_exceedance(results, OUTPUT_DIR)

منحنی **Exceedance (%)** احتمال تجربی *بیشتر بودن* از یک سرعت مشخص را در میان روزهای دارای داده نشان می‌دهد؛ این منحنی معادل دوره بازگشت باد طرح نیست.

## 5. گلباد ۱۰ درجه‌ای و مقایسه جهت همه بادها با بادهای شدید

In [ ]:
wa.plot_windrose(df_dir, results['annual'], OUTPUT_DIR)
wa.plot_direction_comparison(results, OUTPUT_DIR)

## 6. توزیع مشترک سرعت و جهت (Heatmap)

In [ ]:
display(results['joint_percent'].round(2))
wa.plot_joint_distribution(results, OUTPUT_DIR)

## 7. توزیع ماهانه و تغییرات سالانه

In [ ]:
display(results['monthly'].round(2))
wa.plot_monthly_boxplot(df_speed, OUTPUT_DIR)
display(results['annual'].round(2))
wa.plot_annual_variation(results, OUTPUT_DIR)

## 8. ذخیره جداول آماری و داده مرتب‌شده

In [ ]:
wa.export_results(
    df_long, df_speed, df_dir, quality, results,
    output_dir=OUTPUT_DIR
)
wa.print_summary(quality, results)
print('All figures (PNG + PDF), CSV and Excel tables saved to:', OUTPUT_DIR.resolve())

## 9. یادداشت‌های فنی و شیوه انتشار

- نتایج **حداکثر سرعت روزانه** و جهت متناظر را نشان می‌دهند، نه توزیع ساعتی تمام مشاهدات باد.
- جهت باد در سرعت صفر معنی ندارد؛ رکوردهای سرعت صفر در تحلیل سرعت باقی می‌مانند اما از گلباد کنار گذاشته می‌شوند.
- درصد فراوانی از میان رکوردهای معتبر محاسبه می‌شود؛ پوشش داده‌ها را از جدول `Data_Quality` و ستون‌های `Coverage (%)` بررسی کن.
- برای استفاده مهندسی از بادهای حدی یا ادعای روند اقلیمی، ارزیابی تکمیلی منشأ داده، نوع اندازه‌گیری و همگنی زمانی ضروری است.
- برای GitHub، فایل‌های `wind_analysis.py`، `Wind_Analysis.ipynb`، `requirements.txt` و `README.md` را منتشر کن. داده‌های واقعی و پوشه `outputs/` در `.gitignore` قرار دارند.
